# Apple Twitter Sentiment Analysis — Deep Learning & Transformers
**CDS6344 Social Media Computing**

This notebook covers the GPU-heavy models that complement the classical pipeline:
- **Deep Learning:** BiLSTM, CNN-for-text, GRU (Keras / TensorFlow)
- **Transformers:** DistilBERT and BERT, fine-tuned for 3-class sentiment

> Run on **Google Colab with GPU**: `Runtime ▸ Change runtime type ▸ GPU (T4)`.
> Upload `Apple-Twitter-Sentiment-DFE.csv` when prompted, then *Run all*.


In [ ]:
# 1. Install (Colab already has TF & torch; this pins the HF stack)
!pip -q install transformers datasets evaluate scikit-learn --upgrade

In [ ]:
# 2. Imports
import re, html, numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)
sns.set_theme(style='whitegrid')
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 3. Upload the dataset (or use drive)
from google.colab import files
up = files.upload()                      # choose Apple-Twitter-Sentiment-DFE.csv
CSV = next(iter(up))

In [ ]:
# 4. Load + clean (same logic as the classical pipeline)
LABEL_MAP = {'1':'negative','3':'neutral','5':'positive'}
URL = re.compile(r'http\S+|www\.\S+'); MEN = re.compile(r'@\w+')
HASH = re.compile(r'#(\w+)'); NON = re.compile(r'[^a-z\s]')

def clean(t):
    t = html.unescape(str(t)).lower()
    t = URL.sub(' ', t); t = MEN.sub(' ', t); t = HASH.sub(r'\1', t)
    t = re.sub(r'\brt\b',' ',t); t = NON.sub(' ', t)
    return re.sub(r'\s+',' ',t).strip()

df = pd.read_csv(CSV, encoding='latin-1')
df['sentiment'] = df['sentiment'].astype(str).str.strip()
df = df[df['sentiment'].isin(LABEL_MAP)].copy()
df['label'] = df['sentiment'].map(LABEL_MAP)
df['clean'] = df['text'].apply(clean)
df = df[df['clean'].str.len() > 0].reset_index(drop=True)
print(df.shape); print(df['label'].value_counts())

In [ ]:
# 5. Encode labels + split
le = LabelEncoder(); y = le.fit_transform(df['label'])   # alpha order: neg=0,neu=1,pos=2
CLASSES = list(le.classes_)
Xtr, Xte, ytr, yte = train_test_split(df['clean'].values, y, test_size=0.2,
                                      stratify=y, random_state=42)
# class weights for imbalance
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(ytr), y=ytr)
CLASS_WEIGHT = {i:w for i,w in enumerate(cw)}
print(CLASSES, CLASS_WEIGHT)

## Part A — Deep Learning Models (Keras)
Shared embedding + sequence input; three architectures compared.

In [ ]:
# 6. Tokenise + pad for Keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
VOCAB, MAXLEN = 10000, 40
tok = Tokenizer(num_words=VOCAB, oov_token='<OOV>')
tok.fit_on_texts(Xtr)
Xtr_s = pad_sequences(tok.texts_to_sequences(Xtr), maxlen=MAXLEN, padding='post')
Xte_s = pad_sequences(tok.texts_to_sequences(Xte), maxlen=MAXLEN, padding='post')

In [ ]:
# 7. Model builders
from tensorflow.keras import layers, models

def build_bilstm():
    m = models.Sequential([
        layers.Embedding(VOCAB, 128, input_length=MAXLEN),
        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        layers.Bidirectional(layers.LSTM(32)),
        layers.Dropout(0.3), layers.Dense(64, activation='relu'),
        layers.Dropout(0.3), layers.Dense(3, activation='softmax')])
    m.compile('adam','sparse_categorical_crossentropy',metrics=['accuracy']); return m

def build_cnn():
    m = models.Sequential([
        layers.Embedding(VOCAB, 128, input_length=MAXLEN),
        layers.Conv1D(128, 5, activation='relu'),
        layers.GlobalMaxPooling1D(),
        layers.Dense(64, activation='relu'), layers.Dropout(0.3),
        layers.Dense(3, activation='softmax')])
    m.compile('adam','sparse_categorical_crossentropy',metrics=['accuracy']); return m

def build_gru():
    m = models.Sequential([
        layers.Embedding(VOCAB, 128, input_length=MAXLEN),
        layers.Bidirectional(layers.GRU(64)),
        layers.Dropout(0.3), layers.Dense(64, activation='relu'),
        layers.Dropout(0.3), layers.Dense(3, activation='softmax')])
    m.compile('adam','sparse_categorical_crossentropy',metrics=['accuracy']); return m

In [ ]:
# 8. Train + evaluate the three DL models
from tensorflow.keras.callbacks import EarlyStopping
es = EarlyStopping(patience=2, restore_best_weights=True)
dl_results = {}
for name, builder in [('BiLSTM',build_bilstm),('CNN',build_cnn),('GRU',build_gru)]:
    print(f'\n=== {name} ===')
    model = builder()
    model.fit(Xtr_s, ytr, validation_split=0.1, epochs=8, batch_size=64,
              class_weight=CLASS_WEIGHT, callbacks=[es], verbose=2)
    pred = model.predict(Xte_s).argmax(1)
    acc = accuracy_score(yte, pred); f1 = f1_score(yte, pred, average='macro')
    dl_results[name] = {'accuracy':acc,'macro_f1':f1}
    print(f'{name}: acc={acc:.3f} macroF1={f1:.3f}')
    print(classification_report(yte, pred, target_names=CLASSES))

## Part B — Transformer Fine-Tuning (DistilBERT / BERT)
Fine-tunes a pre-trained transformer on the 3-class task. DistilBERT first
(faster); switch `MODEL_NAME` to `bert-base-uncased` or `roberta-base` to compare.

In [ ]:
# 9. Tokenise for the transformer
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from datasets import Dataset
import evaluate

MODEL_NAME = 'distilbert-base-uncased'   # try 'bert-base-uncased' / 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(texts, labels):
    d = Dataset.from_dict({'text':list(texts),'label':list(labels)})
    return d.map(lambda b: tokenizer(b['text'], truncation=True, max_length=64),
                 batched=True)

train_ds, test_ds = to_ds(Xtr, ytr), to_ds(Xte, yte)
collator = DataCollatorWithPadding(tokenizer)

In [ ]:
# 10. Fine-tune
acc_m = evaluate.load('accuracy'); f1_m = evaluate.load('f1')
def metrics(p):
    preds = p.predictions.argmax(-1)
    return {'accuracy':acc_m.compute(predictions=preds, references=p.label_ids)['accuracy'],
            'macro_f1':f1_m.compute(predictions=preds, references=p.label_ids, average='macro')['f1']}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3,
    id2label={i:c for i,c in enumerate(CLASSES)},
    label2id={c:i for i,c in enumerate(CLASSES)})

args = TrainingArguments(output_dir='out', num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01, eval_strategy='epoch',
    save_strategy='no', logging_steps=50, report_to='none')

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
    eval_dataset=test_ds, tokenizer=tokenizer, data_collator=collator,
    compute_metrics=metrics)
trainer.train()
tf_eval = trainer.evaluate(); print(tf_eval)

In [ ]:
# 11. Transformer confusion matrix + report
pred = trainer.predict(test_ds).predictions.argmax(-1)
print(classification_report(yte, pred, target_names=CLASSES))
cm = confusion_matrix(yte, pred)
plt.figure(figsize=(4.5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'{MODEL_NAME} — Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout(); plt.savefig('transformer_cm.png', dpi=130); plt.show()

## Part C — Final Model Comparison
Combine deep-learning and transformer scores into one leaderboard. Paste the
classical-model numbers from the main pipeline to get the full comparison
table for your report.

In [ ]:
# 12. Comparison chart (extend with your classical results)
rows = [{'Model':k,'Accuracy':v['accuracy'],'Macro F1':v['macro_f1']}
        for k,v in dl_results.items()]
rows.append({'Model':MODEL_NAME,'Accuracy':tf_eval['eval_accuracy'],
             'Macro F1':tf_eval['eval_macro_f1']})
comp = pd.DataFrame(rows).sort_values('Macro F1', ascending=False)
print(comp.to_string(index=False))
comp.to_csv('deep_transformer_results.csv', index=False)

ax = comp.set_index('Model')[['Accuracy','Macro F1']].plot(
    kind='bar', figsize=(8,4.5), color=['#264653','#e76f51'])
ax.set_title('Deep Learning & Transformer Comparison'); ax.set_ylim(0,1)
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig('dl_transformer_comparison.png', dpi=130); plt.show()